In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from datetime import date, datetime
from hdbcli import dbapi
from dotenv import load_dotenv
import shutil

warnings.filterwarnings("ignore")

# === CONFIG ===
load_dotenv(dotenv_path="C://Users//ck01976//OneDrive - Crown Paints Kenya PLC//Desktop//Python Scripts//.env")
BASE_DIR = r"C:\Users\ck01976\OneDrive - Crown Paints Kenya PLC\Server_daily sales _report_file"

# === OUTPUT FOLDERS ===
DEALER_DIR = os.path.join(BASE_DIR, "Dealer_Report")
CATEGORY_DIR = os.path.join(BASE_DIR, "Product_Category_Report")
PREMIUM_DIR = os.path.join(BASE_DIR, "Premium_Report")
ARCHIVE_DIR = os.path.join(BASE_DIR, "Archive")

for folder in [DEALER_DIR, CATEGORY_DIR, PREMIUM_DIR, ARCHIVE_DIR]:
    os.makedirs(folder, exist_ok=True)

# === HOLIDAYS (Kenya 2025) ===
from datetime import date
HOLIDAYS = {
    date(2026, 1, 1), date(2026, 4, 18), date(2026, 4, 21),
    date(2025, 6, 1), date(2026, 6, 1), date(2026, 10, 20),
    date(2026, 12, 12), date(2026, 12, 25), date(2026, 12, 26),
}

def is_working_day(d: pd.Timestamp) -> bool:
    return d.weekday() != 6 and d.date() not in HOLIDAYS

def get_last_working_day(ref_date: pd.Timestamp) -> pd.Timestamp:
    d = ref_date - pd.Timedelta(days=1)
    while not is_working_day(d):
        d -= pd.Timedelta(days=1)
    return d

# === AUTO DATE LOGIC ===
TODAY = pd.to_datetime(date.today())
ACT_DATE = get_last_working_day(TODAY)
LY_DATE = ACT_DATE - pd.DateOffset(years=1)
print(f"✅ ACT_DATE: {ACT_DATE.date()} | ✅ LY_DATE: {LY_DATE.date()}")

# === PREMIUM PRODUCTS ===
PREMIUM_TOP5 = [
    'CROWN O/B - SUPER GLOSS',
    'PERMACOTE U/GUARD SILICONE',
    'PERMAPLAST',
    'SILK VINYL EMULSION',
    'VINYL MATT EMULSION'
]

# === EXPORT SALES DATA FROM HANA ===
def export_sales_data():
    conn = dbapi.connect(
        address=os.getenv("HANA_HOST"),
        port=int(os.getenv("HANA_PORT")),
        user=os.getenv("HANA_USER"),
        password=os.getenv("HANA_PASSWORD")
    )
    query = f"""
    SELECT 
        "TerritoryName", "BP SlpName", "CardCode", "CardName", "U_Class",
        "Posting Date", "MajorName", "YTDVolume", "YTDValueWithDocDisc"
    FROM "SBK_CROWN_PAINTS"."AVW_16H_SALES"
    WHERE TO_DATE("Posting Date", 'DD/MM/YYYY')
          BETWEEN TO_DATE('2025-01-01', 'YYYY-MM-DD')
              AND TO_DATE('{ACT_DATE.date()}', 'YYYY-MM-DD')
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

# === UTILITY: SAVE & ARCHIVE ===
def save_and_archive(df, save_path):
    """Save DataFrame to Excel in Sheet1 and archive old version if exists."""
    filename = os.path.basename(save_path)
    if os.path.exists(save_path):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        archive_path = os.path.join(ARCHIVE_DIR, f"{timestamp}_{filename}")
        shutil.copy(save_path, archive_path)
    with pd.ExcelWriter(save_path, engine='openpyxl', mode='w') as writer:
        df.to_excel(writer, index=False, sheet_name="Sheet1")

# === 1️⃣ DEALERWISE REPORT ===
def build_dealer_report(df):
    df["Posting Date"] = pd.to_datetime(df["Posting Date"], errors="coerce")
    df["IsPremium"] = df["MajorName"].isin(PREMIUM_TOP5)
    df["Year"] = df["Posting Date"].dt.year
    df["Month"] = df["Posting Date"].dt.month

    def aggregate_data(data, date):
        mtd = data[(data["Year"] == date.year) & (data["Month"] == date.month) & (data["Posting Date"] <= date)]
        ytd = data[(data["Year"] == date.year) & (data["Posting Date"] <= date)]
        def summarize(d):
            vol, val = d["YTDVolume"].sum(), d["YTDValueWithDocDisc"].sum()
            asp = val / vol if vol else 0
            pvol, pval = d.loc[d["IsPremium"], "YTDVolume"].sum(), d.loc[d["IsPremium"], "YTDValueWithDocDisc"].sum()
            return pd.Series({"Volume": vol, "Value": val, "AvgSP": asp, "Premium_Volume": pvol, "Premium_Value": pval})
        group_cols = ["TerritoryName", "BP SlpName", "CardCode", "CardName", "U_Class"]
        return mtd.groupby(group_cols).apply(summarize).reset_index(), ytd.groupby(group_cols).apply(summarize).reset_index()

    mtd_ly, ytd_ly = aggregate_data(df, LY_DATE)
    mtd_act, ytd_act = aggregate_data(df, ACT_DATE)

    def rename(df_, prefix):
        return df_.rename(columns=lambda x: f"{prefix}_{x}" if x not in ["TerritoryName","BP SlpName","CardCode","CardName","U_Class"] else x)

    mtd_ly, mtd_act, ytd_ly, ytd_act = rename(mtd_ly,"MTD_LY"), rename(mtd_act,"MTD_ACT"), rename(ytd_ly,"YTD_LY"), rename(ytd_act,"YTD_ACT")
    final = mtd_ly.merge(mtd_act,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer") \
                  .merge(ytd_ly,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer") \
                  .merge(ytd_act,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer")

    final.fillna(0,inplace=True)
    def gr(a,b): return (a / b.replace(0,np.nan) - 1) * 100
    final["%GR MTD Volume"] = gr(final["MTD_ACT_Volume"],final["MTD_LY_Volume"])
    final["%GR YTD Volume"] = gr(final["YTD_ACT_Volume"],final["YTD_LY_Volume"])
    final.replace([np.inf,-np.inf],0,inplace=True)
    num_cols = final.select_dtypes(include="number").columns
    final[num_cols] = final[num_cols].round(0)
    return final

# === 2️⃣ PRODUCT CATEGORY REPORT ===
def build_product_category_report(df):
    df['Posting Date'] = pd.to_datetime(df['Posting Date'], errors="coerce")
    df['Year'] = df['Posting Date'].dt.year
    df['Month'] = df['Posting Date'].dt.month

    category_products = {
        'SLAB': ['COVERMATT EMULSION', 'CROWN PUTTIES & FILLERS', 'CROWN TEXTURES FINISHES'],
        'PREM OTHERS': ['ROADLINE PAINTS', 'THINNERS', 'CROWN WOOD FINISHES', 'CROWN U/COATS & PRIMERS'],
        'ECON': ['ECON VESTA RANGE - WB', 'ECON VESTA RANGE - OTHERS', 'ECON VESTA RANGE - O/B', 'ECON VESTA RANGE - VARNISH'],
        'AUTO': ['AUTO PAINT - DUCO - NC', 'AUTO PAINT - DUCO - FAST DRY', 'INDUSTRIAL PAINTS - LOCAL'],
        'PDILITE': ['PIDILITE - FEVICOL ADHE -IMP', 'PIDILITE - DR. FIXIT -IMP', 'PIDILITE - UNITINT - IMP'],
        'OTHERS': ['MODERN TILE ADHESIVE', 'CROWN MISCELLANEOUS', 'LOCAL ADHESIVES']
    }

    mapping = {p: c for c, lst in category_products.items() for p in lst}
    df['Category'] = df['MajorName'].map(mapping).fillna('OTHERS')
    dealer_info = df[['TerritoryName','BP SlpName','CardCode','CardName','U_Class']].drop_duplicates()

    categories = category_products['SLAB'] + ['TOTAL SLAB','PREM OTHERS','ECON','AUTO','PDILITE','OTHERS']
    dealer_product = dealer_info.merge(pd.DataFrame({'Category': categories}), how='cross')

    def compute_ytd(data, date, tag):
        filt = data[(data['Posting Date'] <= date) & (data['Year'] == date.year)]
        grouped = filt.groupby(['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category']) \
                      .agg(Volume=('YTDVolume','sum')).reset_index()
        grouped[f'YTD Vol {tag}'] = grouped['Volume']
        return grouped.drop(columns=['Volume'])

    ytd_act, ytd_ly = compute_ytd(df, ACT_DATE, 'ACT'), compute_ytd(df, LY_DATE, 'LY')

    def calc_total(df_, col):
        total = df_[df_['Category'].isin(category_products['SLAB'])].groupby(
            ['TerritoryName','BP SlpName','CardCode','CardName','U_Class']
        ).agg({col:'sum'}).reset_index()
        total['Category'] = 'TOTAL SLAB'
        return total

    ytd_act = pd.concat([ytd_act, calc_total(ytd_act,'YTD Vol ACT')])
    ytd_ly = pd.concat([ytd_ly, calc_total(ytd_ly,'YTD Vol LY')])

    merged = dealer_product.merge(ytd_ly,on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category'],how='left') \
                           .merge(ytd_act,on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category'],how='left') \
                           .fillna(0)
    merged['% GR'] = np.where(merged['YTD Vol LY']>0,(merged['YTD Vol ACT']/merged['YTD Vol LY']-1)*100,0)
    merged.rename(columns={'YTD Vol LY':'L/YR','YTD Vol ACT':'ACT'},inplace=True)
    return merged

# === 3️⃣ PREMIUM TOP 5 REPORT ===
# (Your logic retained exactly as before)
def build_premium_report(df):
    df['Posting Date'] = pd.to_datetime(df['Posting Date'])
    df['Year'] = df['Posting Date'].dt.year
    df['Month'] = df['Posting Date'].dt.month
    dealer_info = df[['TerritoryName','BP SlpName','CardCode','CardName','U_Class']].drop_duplicates()
    product_df = pd.DataFrame({'MajorName': PREMIUM_TOP5 + ['TOP 5 TOTAL']})
    dealer_product = dealer_info.merge(product_df, how='cross')
    def compute_metrics(data, date, tag):
        filt = data[(data['Posting Date'] <= date) & (data['MajorName'].isin(PREMIUM_TOP5))]
        def summarize(mask, label):
            grouped = filt[mask].groupby(['TerritoryName','BP SlpName','CardCode','CardName','U_Class','MajorName']) \
                .agg(Volume=('YTDVolume','sum')).reset_index()
            grouped[f'{label} Volume {tag}'] = grouped['Volume']
            return grouped.drop(columns='Volume')
        mtd_mask = (filt['Year']==date.year)&(filt['Month']==date.month)
        ytd_mask = (filt['Year']==date.year)
        mtd, ytd = summarize(mtd_mask,'MTD'), summarize(ytd_mask,'YTD')
        def totalize(df_part,label):
            total = df_part.groupby(['TerritoryName','BP SlpName','CardCode','CardName','U_Class']) \
                .agg({f'{label} Volume {tag}':'sum'}).reset_index()
            total['MajorName'] = 'TOP 5 TOTAL'
            return total
        return pd.concat([mtd,totalize(mtd,'MTD')]), pd.concat([ytd,totalize(ytd,'YTD')])
    mtd_act, ytd_act = compute_metrics(df, ACT_DATE, 'ACT')
    mtd_ly, ytd_ly = compute_metrics(df, LY_DATE, 'LY')
    merged = dealer_product.copy()
    for d in [mtd_ly,mtd_act,ytd_ly,ytd_act]:
        merged = merged.merge(d,on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class','MajorName'],how='left')
    merged.fillna(0,inplace=True)
    def reshape(df,time_tag):
        reshaped = pd.DataFrame()
        for prod in PREMIUM_TOP5+['TOP 5 TOTAL']:
            vol_ly,vol_act=f'{time_tag} Volume LY',f'{time_tag} Volume ACT'
            sub = df[df['MajorName']==prod][['TerritoryName','BP SlpName','CardCode','CardName','U_Class',vol_ly,vol_act]]
            sub.columns=['TerritoryName','BP SlpName','CardCode','CardName','U_Class',f'{prod} {time_tag} Vol LY',f'{prod} {time_tag} Vol ACT']
            reshaped=sub if reshaped.empty else reshaped.merge(sub,on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class'],how='outer')
        return reshaped
    mtd, ytd = reshape(merged,'MTD'), reshape(merged,'YTD')
    premium_report = mtd.merge(ytd,on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class']).fillna(0)
    for prod in PREMIUM_TOP5+['TOP 5 TOTAL']:
        for tag in ['MTD','YTD']:
            act, ly = f'{prod} {tag} Vol ACT', f'{prod} {tag} Vol LY'
            premium_report[f'{prod} {tag} Vol G/R'] = np.where(premium_report[ly]>0,(premium_report[act]/premium_report[ly]-1)*100,0)
    num_cols = premium_report.select_dtypes(include='number').columns
    premium_report[num_cols] = premium_report[num_cols].round(0)
    return premium_report

# === MAIN EXECUTION ===
if __name__ == "__main__":
    print("🔎 Fetching sales data...")
    df = export_sales_data()
    print(f"✅ Data loaded: {df.shape}")

    # Dealer Report
    dealer_report = build_dealer_report(df)
    dealer_file = os.path.join(DEALER_DIR, "Dealer_Report.xlsx")
    save_and_archive(dealer_report, dealer_file)
    print(f"📂 Dealerwise report saved → {dealer_file}")

    # Product Category Report
    cat_report = build_product_category_report(df)
    cat_file = os.path.join(CATEGORY_DIR, "Product_Category_Report.xlsx")
    save_and_archive(cat_report, cat_file)
    print(f"📂 Product category report saved → {cat_file}")

    # Premium Report
    premium_report = build_premium_report(df)
    premium_file = os.path.join(PREMIUM_DIR, "Premium_Report.xlsx")
    save_and_archive(premium_report, premium_file)
    print(f"📂 Premium Top 5 report saved → {premium_file}")

    print("✅ All three reports successfully generated, saved, and archived.")


✅ ACT_DATE: 2026-01-23 | ✅ LY_DATE: 2025-01-23
🔎 Fetching sales data...
✅ Data loaded: (1069435, 9)


ValueError: cannot insert U_Class, already exists

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
from datetime import date
from hdbcli import dbapi
from dotenv import load_dotenv
from openpyxl import load_workbook

# Suppress warnings
warnings.filterwarnings("ignore")

# === CONFIG ===
load_dotenv("C://Users//ck01976//OneDrive - Crown Paints Kenya PLC//Desktop//Python Scripts//.env")

BASE_DIR = r"C:\Users\ck01976\OneDrive - Crown Paints Kenya PLC\Server_daily sales _report_file"
DEALERWISE_PATH = os.path.join(BASE_DIR, "Dealerwise_Report", "Dealerwise_Report.xlsx")
PREMIUM_PATH = os.path.join(BASE_DIR, "Premium_Report", "Premium_Report.xlsx")
PRODUCT_PATH = os.path.join(BASE_DIR, "Product_Category", "Product_Category_Report.xlsx")
ARCHIVE_DIR = os.path.join(BASE_DIR, "Archive")

os.makedirs(ARCHIVE_DIR, exist_ok=True)

# === HOLIDAYS ===
HOLIDAYS = {
    date(2025, 1, 1), date(2025, 4, 18), date(2025, 4, 21),
    date(2025, 5, 1), date(2025, 6, 1), date(2025, 10, 20),
    date(2025, 12, 12), date(2025, 12, 25), date(2025, 12, 26)
}

def is_working_day(d):
    return d.weekday() != 6 and d.date() not in HOLIDAYS

def get_last_working_day(ref_date):
    d = ref_date - pd.Timedelta(days=1)
    while not is_working_day(d):
        d -= pd.Timedelta(days=1)
    return d

TODAY = pd.to_datetime(date.today())
ACT_DATE = get_last_working_day(TODAY)
LY_DATE = ACT_DATE - pd.DateOffset(years=1)

print(f"✅ ACT_DATE: {ACT_DATE.date()} | ✅ LY_DATE: {LY_DATE.date()}")

PREMIUM_TOP5 = [
    "CROWN O/B - SUPER GLOSS",
    "PERMACOTE U/GUARD SILICONE",
    "PERMAPLAST",
    "SILK VINYL EMULSION",
    "VINYL MATT EMULSION"
]

# === DATABASE EXPORT ===
def export_sales_data():
    conn = dbapi.connect(
        address=os.getenv("HANA_HOST"),
        port=int(os.getenv("HANA_PORT")),
        user=os.getenv("HANA_USER"),
        password=os.getenv("HANA_PASSWORD")
    )
    query = f"""
    SELECT "TerritoryName","BP SlpName","CardCode","CardName","U_Class",
           "Posting Date","MajorName","YTDVolume","YTDValueWithDocDisc"
    FROM "SBK_CROWN_PAINTS"."AVW_16H_SALES"
    WHERE TO_DATE("Posting Date",'DD/MM/YYYY') 
          BETWEEN TO_DATE('2024-01-01','YYYY-MM-DD') 
              AND TO_DATE('{ACT_DATE.date()}','YYYY-MM-DD')
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

# === COMMON SAVE FUNCTION ===
def save_to_excel(df, path, archive_name):
    # Overwrite Sheet1 only
    with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df.to_excel(writer, index=False, sheet_name="Sheet1")
    print(f"💾 Updated Sheet1 in: {path}")

    # Archive copy
    archive_file = os.path.join(ARCHIVE_DIR, f"{archive_name}_{ACT_DATE.date()}.xlsx")
    df.to_excel(archive_file, index=False)
    print(f"📦 Archived: {archive_file}")

# === DEALERWISE REPORT ===
def calculate_card_metrics(data, target_date):
    df = data.copy()
    df["IsPremium"] = df["MajorName"].isin(PREMIUM_TOP5)
    df["Year"] = df["Posting Date"].dt.year
    df["Month"] = df["Posting Date"].dt.month

    mtd_data = df[
        (df["Year"] == target_date.year) & 
        (df["Month"] == target_date.month) &
        (df["Posting Date"] <= target_date)
    ]
    ytd_data = df[
        (df["Year"] == target_date.year) &
        (df["Posting Date"] <= target_date)
    ]

    group_cols = ["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]

    def agg_metrics(d):
        vol = d["YTDVolume"].sum()
        val = d["YTDValueWithDocDisc"].sum()
        asp = val / vol if vol else 0
        pvol = d.loc[d["IsPremium"],"YTDVolume"].sum()
        pval = d.loc[d["IsPremium"],"YTDValueWithDocDisc"].sum()
        return {"Volume":vol,"Value":val,"AvgSP":asp,"Premium_Volume":pvol,"Premium_Value":pval}

    mtd = mtd_data.groupby(group_cols).apply(agg_metrics).apply(pd.Series).reset_index()
    ytd = ytd_data.groupby(group_cols).apply(agg_metrics).apply(pd.Series).reset_index()
    return mtd, ytd

def build_dealer_report(df):
    df["Posting Date"] = pd.to_datetime(df["Posting Date"], errors="coerce")
    df["YTDValueWithDocDisc"] = pd.to_numeric(df["YTDValueWithDocDisc"], errors="coerce")
    df["YTDVolume"] = pd.to_numeric(df["YTDVolume"], errors="coerce")

    mtd_ly, ytd_ly = calculate_card_metrics(df, LY_DATE)
    mtd_act, ytd_act = calculate_card_metrics(df, ACT_DATE)

    def rename_cols(x,prefix):
        return x.rename(columns=lambda c: f"{prefix}_{c}" if c not in ["TerritoryName","BP SlpName","CardCode","CardName","U_Class"] else c)

    mtd_ly, mtd_act = rename_cols(mtd_ly,"MTD_LY"), rename_cols(mtd_act,"MTD_ACT")
    ytd_ly, ytd_act = rename_cols(ytd_ly,"YTD_LY"), rename_cols(ytd_act,"YTD_ACT")

    dfm = mtd_ly.merge(mtd_act,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer")
    dfm = dfm.merge(ytd_ly,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer")
    dfm = dfm.merge(ytd_act,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer")

    dfm.fillna(0,inplace=True)
    def pct(act,ly): return (act/ly.replace(0,np.nan)-1)*100

    dfm["%GR MTD Volume"]=pct(dfm["MTD_ACT_Volume"],dfm["MTD_LY_Volume"])
    dfm["%GR YTD Volume"]=pct(dfm["YTD_ACT_Volume"],dfm["YTD_LY_Volume"])
    dfm.replace([np.inf,-np.inf],0,inplace=True)

    return dfm.round(0)

# === PREMIUM REPORT ===
def build_premium_report(df):
    df["Posting Date"] = pd.to_datetime(df["Posting Date"])
    df["Year"], df["Month"] = df["Posting Date"].dt.year, df["Posting Date"].dt.month

    dealer_info = df[["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]].drop_duplicates()
    product_df = pd.DataFrame({"MajorName": PREMIUM_TOP5 + ["TOP 5 TOTAL"]})
    dealer_product = dealer_info.merge(product_df, how="cross")

    def compute_metrics(data,date,tag):
        f = data[(data["Posting Date"]<=date)&(data["MajorName"].isin(PREMIUM_TOP5))]
        def summarize(mask,label):
            g=f[mask].groupby(["TerritoryName","BP SlpName","CardCode","CardName","U_Class","MajorName"])\
                .agg(Volume=("YTDVolume","sum")).reset_index()
            g[f"{label} Volume {tag}"]=g["Volume"]
            return g.drop(columns="Volume")
        mtd_mask=(f["Year"]==date.year)&(f["Month"]==date.month)
        ytd_mask=(f["Year"]==date.year)
        mtd=summarize(mtd_mask,"MTD"); ytd=summarize(ytd_mask,"YTD")
        def totalize(df_,label):
            t=df_.groupby(["TerritoryName","BP SlpName","CardCode","CardName","U_Class"])\
                .agg({f"{label} Volume {tag}":"sum"}).reset_index()
            t["MajorName"]="TOP 5 TOTAL"; return t
        return pd.concat([mtd,totalize(mtd,"MTD")]), pd.concat([ytd,totalize(ytd,"YTD")])

    mtd_act,ytd_act=compute_metrics(df,ACT_DATE,"ACT")
    mtd_ly,ytd_ly=compute_metrics(df,LY_DATE,"LY")

    merged=dealer_product.copy()
    for d in [mtd_ly,mtd_act,ytd_ly,ytd_act]:
        merged=merged.merge(d,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class","MajorName"],how="left")
    merged.fillna(0,inplace=True)

    def reshape(df_,tag):
        out=pd.DataFrame()
        for p in PREMIUM_TOP5+["TOP 5 TOTAL"]:
            ly,act=f"{tag} Volume LY",f"{tag} Volume ACT"
            sub=df_[df_["MajorName"]==p][["TerritoryName","BP SlpName","CardCode","CardName","U_Class",ly,act]].copy()
            sub.columns=["TerritoryName","BP SlpName","CardCode","CardName","U_Class",f"{p} {tag} Vol LY",f"{p} {tag} Vol ACT"]
            out=sub if out.empty else out.merge(sub,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],how="outer")
        return out

    mtd=reshape(merged,"MTD"); ytd=reshape(merged,"YTD")
    report=pd.merge(mtd,ytd,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]).fillna(0)

    for p in PREMIUM_TOP5+["TOP 5 TOTAL"]:
        for tag in ["MTD","YTD"]:
            act,ly=f"{p} {tag} Vol ACT",f"{p} {tag} Vol LY"
            report[f"{p} {tag} Vol G/R"]=report.apply(lambda r:((r[act]/r[ly])-1)*100 if r[ly]!=0 else 0,axis=1)

    base=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]
    cols=base+[f"{p} {t} Vol {x}" for p in PREMIUM_TOP5+["TOP 5 TOTAL"] for t in ["MTD","YTD"] for x in ["LY","ACT","G/R"]]
    report=report[cols]; report[report.select_dtypes(include="number").columns]=report.select_dtypes(include="number").round(0)
    return report

# === PRODUCT CATEGORY REPORT ===
def build_product_category_report(df):
    df["Posting Date"]=pd.to_datetime(df["Posting Date"],errors="coerce")
    df["Year"],df["Month"]=df["Posting Date"].dt.year,df["Posting Date"].dt.month
    category_products={
        "SLAB":["COVERMATT EMULSION","CROWN PUTTIES & FILLERS","CROWN TEXTURES FINISHES"],
        "PREM OTHERS":["ROADLINE PAINTS","THINNERS","CROWN WOOD FINISHES","CROWN U/COATS & PRIMERS"],
        "ECON":["ECON VESTA RANGE - WB","ECON VESTA RANGE - OTHERS","ECON VESTA RANGE - O/B","ECON VESTA RANGE - VARNISH"],
        "AUTO":["AUTO PAINT - DUCO - NC","AUTO PAINT - DUCO - FAST DRY","INDUSTRIAL PAINTS - LOCAL"],
        "PDILITE":["PIDILITE - FEVICOL ADHE -IMP","PIDILITE - DR. FIXIT -IMP","PIDILITE - UNITINT - IMP"],
        "OTHERS":["MODERN TILE ADHESIVE","CROWN MISCELLANEOUS","LOCAL ADHESIVES"]
    }
    mapping={p:c for c,v in category_products.items() for p in v}
    df["Category"]=df["MajorName"].map(mapping).fillna("OTHERS")
    dealer=df[["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]].drop_duplicates()
    cats=category_products["SLAB"]+["TOTAL SLAB","PREM OTHERS","ECON","AUTO","PDILITE","OTHERS"]
    dp=dealer.merge(pd.DataFrame({"Category":cats}),how="cross")

    def compute(data,date,tag):
        f=data[(data["Posting Date"]<=date)&(data["Year"]==date.year)]
        g=f.groupby(["TerritoryName","BP SlpName","CardCode","CardName","U_Class","Category"])\
            .agg(Volume=("YTDVolume","sum")).reset_index()
        g[f"YTD Vol {tag}"]=g["Volume"]; return g.drop(columns="Volume")

    act,ly=compute(df,ACT_DATE,"ACT"),compute(df,LY_DATE,"LY")

    def total_slab(x,col):
        t=x[x["Category"].isin(category_products["SLAB"])].groupby(["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]).agg({col:"sum"}).reset_index()
        t["Category"]="TOTAL SLAB"; return t
    act=pd.concat([act,total_slab(act,"YTD Vol ACT")]); ly=pd.concat([ly,total_slab(ly,"YTD Vol LY")])
    m=dp.merge(ly,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class","Category"],how="left")\
        .merge(act,on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class","Category"],how="left").fillna(0)
    m["% GR"]=m.apply(lambda r:((r["YTD Vol ACT"]/r["YTD Vol LY"])-1)*100 if r["YTD Vol LY"]!=0 else 0,axis=1)
    m.rename(columns={"YTD Vol LY":"L/YR","YTD Vol ACT":"ACT"},inplace=True)
    pvt=m.pivot(index=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"],columns="Category",values=["L/YR","ACT","% GR"])
    pvt.columns=[f"{c2} {c1}" for c1,c2 in pvt.columns]; pvt.reset_index(inplace=True)
    base=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"]
    final_cats=["COVERMATT EMULSION","CROWN PUTTIES & FILLERS","CROWN TEXTURES FINISHES","TOTAL SLAB","PREM OTHERS","ECON","OTHERS","AUTO","PDILITE"]
    order=base+[f"{c} {m}" for c in final_cats for m in ["L/YR","ACT","% GR"]]
    for c in order:
        if c not in pvt.columns: pvt[c]=0
    return pvt[order]

# === MAIN EXECUTION ===
if __name__ == "__main__":
    print("🔎 Fetching sales data from SAP HANA...")
    df = export_sales_data()
    print(f"✅ Data loaded: {df.shape}")

    # Dealerwise
    dealer_report = build_dealer_report(df)
    save_to_excel(dealer_report, DEALERWISE_PATH, "Dealerwise_Archive")

    # Premium Top-5
    premium_report = build_premium_report(df)
    save_to_excel(premium_report, PREMIUM_PATH, "Premium_Archive")

    # Product Category
    category_report = build_product_category_report(df)
    save_to_excel(category_report, PRODUCT_PATH, "ProductCategory_Archive")

    print("🎯 All reports updated and archived successfully.")


✅ ACT_DATE: 2026-01-23 | ✅ LY_DATE: 2025-01-23
🔎 Fetching sales data from SAP HANA...
✅ Data loaded: (1069435, 9)


ValueError: cannot insert U_Class, already exists

In [3]:
import os
import warnings
import pandas as pd
import numpy as np
from datetime import date
from hdbcli import dbapi
from dotenv import load_dotenv

# Suppress warnings
warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv(dotenv_path="C://Users//ck01976//OneDrive - Crown Paints Kenya PLC//Desktop//Python Scripts//.env")

# === CONFIG ===
OUTPUT_DIR = r"C:\Users\ck01976\OneDrive - Crown Paints Kenya PLC\Server_daily sales _report_file"

# Holidays (Kenya 2025, add more as needed)
HOLIDAYS = {
    date(2025, 1, 1),   # New Year
    date(2025, 4, 18),  # Good Friday
    date(2025, 4, 21),  # Easter Monday
    date(2025, 5, 1),   # Labour Day
    date(2025, 6, 1),   # Madaraka Day
    date(2025, 10, 20), # Mashujaa Day
    date(2025, 12, 12), # Jamhuri Day
    date(2025, 12, 25), # Christmas
    date(2025, 12, 26), # Boxing Day
}

def is_working_day(d: pd.Timestamp) -> bool:
    return d.weekday() != 6 and d.date() not in HOLIDAYS  # exclude Sundays + holidays

def get_last_working_day(ref_date: pd.Timestamp) -> pd.Timestamp:
    """Find the most recent working day before ref_date"""
    d = ref_date - pd.Timedelta(days=1)
    while not is_working_day(d):
        d -= pd.Timedelta(days=1)
    return d

# === AUTO DATE LOGIC ===
TODAY = pd.to_datetime(date.today())
ACT_DATE = get_last_working_day(TODAY)
LY_DATE = ACT_DATE - pd.DateOffset(years=1)

print(f"✅ ACT_DATE: {ACT_DATE.date()} | ✅ LY_DATE: {LY_DATE.date()}")

# Premium Top 5 list
PREMIUM_TOP5 = [
    "CROWN O/B - SUPER GLOSS",
    "PERMACOTE U/GUARD SILICONE",
    "PERMAPLAST",
    "SILK VINYL EMULSION",
    "VINYL MATT EMULSION",
]

# === EXPORT SALES DATA FROM HANA ===
def export_sales_data():
    conn = dbapi.connect(
        address=os.getenv("HANA_HOST"),
        port=int(os.getenv("HANA_PORT")),
        user=os.getenv("HANA_USER"),
        password=os.getenv("HANA_PASSWORD")
    )
    query = f"""
    SELECT 
        "TerritoryName",
        "BP SlpName",
        "CardCode",
        "CardName",
        "U_Class",
        "Posting Date",
        "MajorName",
        "YTDVolume",
        "YTDValueWithDocDisc"
    FROM "SBK_CROWN_PAINTS"."AVW_16H_SALES"
    WHERE TO_DATE("Posting Date", 'DD/MM/YYYY') 
          BETWEEN TO_DATE('2024-01-01', 'YYYY-MM-DD') 
              AND TO_DATE('{ACT_DATE.date()}', 'YYYY-MM-DD')
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

# === DEALERWISE REPORT ===
def calculate_card_metrics(data, target_date):
    df = data.copy()
    df["IsPremium"] = df["MajorName"].isin(PREMIUM_TOP5)
    df["Year"] = df["Posting Date"].dt.year
    df["Month"] = df["Posting Date"].dt.month

    mtd_data = df[
        (df["Year"] == target_date.year) &
        (df["Month"] == target_date.month) &
        (df["Posting Date"] <= target_date)
    ]
    ytd_data = df[
        (df["Year"] == target_date.year) &
        (df["Posting Date"] <= target_date)
    ]

    group_cols = ["TerritoryName", "BP SlpName", "CardCode", "CardName", "U_Class"]

    def agg_metrics(d):
        volume = d["YTDVolume"].sum()
        value = d["YTDValueWithDocDisc"].sum()
        avgsp = value / volume if volume else 0
        prem_vol = d.loc[d["IsPremium"], "YTDVolume"].sum()
        prem_val = d.loc[d["IsPremium"], "YTDValueWithDocDisc"].sum()
        return {
            "Volume": volume,
            "Value": value,
            "AvgSP": avgsp,
            "Premium_Volume": prem_vol,
            "Premium_Value": prem_val,
        }

    mtd = mtd_data.groupby(group_cols).apply(agg_metrics).apply(pd.Series).reset_index()
    ytd = ytd_data.groupby(group_cols).apply(agg_metrics).apply(pd.Series).reset_index()
    return mtd, ytd

def rename_columns(df, prefix):
    return df.rename(columns=lambda x: f"{prefix}_{x}" if x not in ["TerritoryName","BP SlpName","CardCode","CardName","U_Class"] else x)

def percent_growth(act, ly):
    return (act / ly.replace(0, np.nan) - 1) * 100

def build_dealer_report(df):
    df["Posting Date"] = pd.to_datetime(df["Posting Date"], dayfirst=True, errors="coerce")
    df["YTDValueWithDocDisc"] = pd.to_numeric(df["YTDValueWithDocDisc"], errors="coerce")
    df["YTDVolume"] = pd.to_numeric(df["YTDVolume"], errors="coerce")

    mtd_ly, ytd_ly = calculate_card_metrics(df, LY_DATE)
    mtd_act, ytd_act = calculate_card_metrics(df, ACT_DATE)

    mtd_ly = rename_columns(mtd_ly, "MTD_LY")
    mtd_act = rename_columns(mtd_act, "MTD_ACT")
    ytd_ly = rename_columns(ytd_ly, "YTD_LY")
    ytd_act = rename_columns(ytd_act, "YTD_ACT")

    final = mtd_ly.merge(mtd_act, on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"], how="outer")
    final = final.merge(ytd_ly, on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"], how="outer")
    final = final.merge(ytd_act, on=["TerritoryName","BP SlpName","CardCode","CardName","U_Class"], how="outer")

    final.fillna(0, inplace=True)

    final["%GR MTD Volume"] = percent_growth(final["MTD_ACT_Volume"], final["MTD_LY_Volume"])
    final["%GR MTD Value"] = percent_growth(final["MTD_ACT_Value"], final["MTD_LY_Value"])
    final["%Change MTD AvgSP"] = final["MTD_ACT_AvgSP"] - final["MTD_LY_AvgSP"]
    final["%GR MTD Premium Volume"] = percent_growth(final["MTD_ACT_Premium_Volume"], final["MTD_LY_Premium_Volume"])
    final["%GR MTD Premium Value"] = percent_growth(final["MTD_ACT_Premium_Value"], final["MTD_LY_Premium_Value"])

    final["%GR YTD Volume"] = percent_growth(final["YTD_ACT_Volume"], final["YTD_LY_Volume"])
    final["%GR YTD Value"] = percent_growth(final["YTD_ACT_Value"], final["YTD_LY_Value"])
    final["%Change YTD AvgSP"] = final["YTD_ACT_AvgSP"] - final["YTD_LY_AvgSP"]
    final["%GR YTD Premium Volume"] = percent_growth(final["YTD_ACT_Premium_Volume"], final["YTD_LY_Premium_Volume"])
    final["%GR YTD Premium Value"] = percent_growth(final["YTD_ACT_Premium_Value"], final["YTD_LY_Premium_Value"])

    final.replace([np.inf, -np.inf], 0, inplace=True)
    final.fillna(0, inplace=True)

    num_cols = final.select_dtypes(include="number").columns
    final[num_cols] = final[num_cols].round(0)

    final_report = final[[
        "BP SlpName","TerritoryName","CardCode","CardName","U_Class",
        "MTD_LY_Volume","MTD_ACT_Volume","%GR MTD Volume",
        "MTD_LY_Value","MTD_ACT_Value","%GR MTD Value",
        "MTD_LY_AvgSP","MTD_ACT_AvgSP","%Change MTD AvgSP",
        "MTD_LY_Premium_Volume","MTD_ACT_Premium_Volume","%GR MTD Premium Volume",
        "MTD_LY_Premium_Value","MTD_ACT_Premium_Value","%GR MTD Premium Value",
        "YTD_LY_Volume","YTD_ACT_Volume","%GR YTD Volume",
        "YTD_LY_Value","YTD_ACT_Value","%GR YTD Value",
        "YTD_LY_AvgSP","YTD_ACT_AvgSP","%Change YTD AvgSP",
        "YTD_LY_Premium_Volume","YTD_ACT_Premium_Volume","%GR YTD Premium Volume",
        "YTD_LY_Premium_Value","YTD_ACT_Premium_Value","%GR YTD Premium Value",
    ]]
    return final_report

# === PRODUCT CATEGORY REPORT ===
def build_product_category_report(df):
    df['Posting Date'] = pd.to_datetime(df['Posting Date'], dayfirst=True, errors="coerce")
    df['Year'] = df['Posting Date'].dt.year
    df['Month'] = df['Posting Date'].dt.month

    # Category mapping
    category_products = {
        'SLAB': ['COVERMATT EMULSION', 'CROWN PUTTIES & FILLERS', 'CROWN TEXTURES FINISHES'],
        'PREM OTHERS': ['ROADLINE PAINTS', 'THINNERS', 'CROWN WOOD FINISHES', 'CROWN U/COATS & PRIMERS'],
        'ECON': ['ECON VESTA RANGE - WB', 'ECON VESTA RANGE - OTHERS', 'ECON VESTA RANGE - O/B', 'ECON VESTA RANGE - VARNISH'],
        'AUTO': ['AUTO PAINT - DUCO - NC', 'AUTO PAINT - DUCO - FAST DRY', 'INDUSTRIAL PAINTS - LOCAL'],
        'PDILITE': ['PIDILITE - FEVICOL ADHE -IMP', 'PIDILITE - DR. FIXIT -IMP', 'PIDILITE - UNITINT - IMP'],
        'OTHERS': ['MODERN TILE ADHESIVE', 'CROWN MISCELLANEOUS', 'LOCAL ADHESIVES']
    }
    product_to_category = {prod: cat for cat, prods in category_products.items() for prod in prods}
    df['Category'] = df['MajorName'].map(product_to_category)
    df.loc[df['MajorName'].isin(category_products['SLAB']), 'Category'] = df['MajorName']
    df['Category'] = df['Category'].fillna('OTHERS')

    dealer_info = df[['TerritoryName','BP SlpName','CardCode','CardName','U_Class']].drop_duplicates()
    slab_products = category_products['SLAB']
    categories = slab_products + ['TOTAL SLAB', 'PREM OTHERS','ECON','AUTO','PDILITE','OTHERS']
    dealer_product = dealer_info.merge(pd.DataFrame({'Category': categories}), how='cross')

    def compute_ytd(data, date, tag):
        filtered = data[(data['Posting Date'] <= date) & (data['Year'] == date.year)]
        grouped = filtered.groupby(['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category']) \
            .agg(Volume=('YTDVolume','sum')).reset_index()
        grouped[f'YTD Vol {tag}'] = grouped['Volume']
        return grouped.drop(columns=['Volume'])

    ytd_act = compute_ytd(df, ACT_DATE, 'ACT')
    ytd_ly = compute_ytd(df, LY_DATE, 'LY')

    def calc_total_slab(df_, col):
        total = df_[df_['Category'].isin(slab_products)].groupby(
            ['TerritoryName','BP SlpName','CardCode','CardName','U_Class']
        ).agg({col:'sum'}).reset_index()
        total['Category'] = 'TOTAL SLAB'
        return total

    ytd_act = pd.concat([ytd_act, calc_total_slab(ytd_act,'YTD Vol ACT')], ignore_index=True)
    ytd_ly = pd.concat([ytd_ly, calc_total_slab(ytd_ly,'YTD Vol LY')], ignore_index=True)

    merged = dealer_product.copy()
    merged = merged.merge(ytd_ly, on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category'], how='left')
    merged = merged.merge(ytd_act, on=['TerritoryName','BP SlpName','CardCode','CardName','U_Class','Category'], how='left')
    merged.fillna(0, inplace=True)

    merged['% GR'] = merged.apply(
        lambda row: ((row['YTD Vol ACT'] / row['YTD Vol LY'] - 1) if row['YTD Vol LY'] != 0 else 0), axis=1
    )

    merged.rename(columns={'YTD Vol LY':'L/YR','YTD Vol ACT':'ACT'}, inplace=True)

    report = merged.pivot_table(
        index=['TerritoryName','BP SlpName','CardCode','CardName','U_Class'],
        columns='Category',
        values=['L/YR','ACT','% GR']
    )
    report.columns = [f'{cat} {metric}' for metric, cat in report.columns]
    report.reset_index(inplace=True)

    base_cols = ['TerritoryName','BP SlpName','CardCode','CardName','U_Class']
    final_categories = [
        'COVERMATT EMULSION','CROWN PUTTIES & FILLERS','CROWN TEXTURES FINISHES',
        'TOTAL SLAB','PREM OTHERS','ECON','OTHERS','AUTO','PDILITE'
    ]
    metrics = ['L/YR','ACT','% GR']
    ordered_columns = base_cols + [f'{cat} {metric}' for cat in final_categories for metric in metrics]

    for col in ordered_columns:
        if col not in report.columns:
            report[col] = 0

    report = report[ordered_columns]
    return report

# === MAIN ===
if __name__ == "__main__":
    print("🔎 Fetching sales data...")
    df = export_sales_data()
    print("✅ Data loaded:", df.shape)

    # Dealerwise Report
    dealer_report = build_dealer_report(df)
    dealer_file = os.path.join(OUTPUT_DIR, f"Dealerwise_Report_{ACT_DATE.date()}.xlsx")
    dealer_report.to_excel(dealer_file, index=False)
    print(f"📂 Dealerwise report saved to {dealer_file}")

    # Product Category Report
    cat_report = build_product_category_report(df)
    cat_file = os.path.join(OUTPUT_DIR, f"Product_Category_Report_{ACT_DATE.date()}.xlsx")
    cat_report.to_excel(cat_file, index=False)
    print(f"📂 Product category report saved to {cat_file}")


✅ ACT_DATE: 2026-01-23 | ✅ LY_DATE: 2025-01-23
🔎 Fetching sales data...
✅ Data loaded: (1069435, 9)


ValueError: cannot insert U_Class, already exists